# Transfer Learning: ResNet50 on Dogs vs Cats

Compare frozen-base **feature extraction** vs **fine-tuning** the last 30 layers.

## Section 0: Environment Check (Colab-friendly)

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
    print("Running in Google Colab.")
    print("Use a GPU runtime: Runtime > Change runtime type > GPU.")
except ImportError:
    IN_COLAB = False
    print("Not running in Colab.")
    print("Local execution is possible, but ResNet50 training is slow without a GPU.")

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Project root:", project_root)

### Kaggle API setup (run in Colab after uploading `kaggle.json`)

1. Create a token at [kaggle.com/settings](https://www.kaggle.com/settings) → **API → Create New Token**.
2. Upload `kaggle.json` to the Colab working directory.
3. Run:

```python
!pip install -q kaggle
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c dogs-vs-cats -p dataset/raw
!unzip -q dataset/raw/dogs-vs-cats.zip -d dataset/raw
!unzip -q dataset/raw/train.zip -d dataset/raw
```

## Section 1: Organize Dataset

In [ ]:
import random
import shutil

SEED = 42
random.seed(SEED)

raw_train = project_root / "dataset" / "raw" / "train"
if not raw_train.exists():
    raise FileNotFoundError(
        f"Expected images in {raw_train}. "
        "Download/unzip Dogs vs Cats first (see Section 0)."
    )

image_files = list(raw_train.glob("*.jpg"))
cats = [p for p in image_files if p.name.startswith("cat.")]
dogs = [p for p in image_files if p.name.startswith("dog.")]


def split_and_copy(files, class_name):
    files = files.copy()
    random.shuffle(files)
    n = len(files)
    n_train = int(0.80 * n)
    n_val = int(0.10 * n)
    splits = {
        "train": files[:n_train],
        "val": files[n_train : n_train + n_val],
        "test": files[n_train + n_val :],
    }
    counts = {}
    for split_name, items in splits.items():
        dest = project_root / "dataset" / split_name / class_name
        dest.mkdir(parents=True, exist_ok=True)
        for src in items:
            shutil.copy(src, dest / src.name)
        counts[split_name] = len(items)
        print(f"{split_name}/{class_name}: {len(items)}")
    return counts


print("Organizing dataset (80/10/10, seed=42)...")
print("--- cats ---")
split_and_copy(cats, "cats")
print("--- dogs ---")
split_and_copy(dogs, "dogs")

## Section 2: Data Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    project_root / "dataset" / "train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
)
val_generator = val_datagen.flow_from_directory(
    project_root / "dataset" / "val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
)

print("class_indices:", train_generator.class_indices)

## Section 3: Feature Extraction (Frozen Base)

In [ ]:
import time

from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
)
base_model.trainable = False

model = models.Sequential(
    [
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer=Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

print("Starting feature-extraction training (frozen ResNet50 base)...")
t0 = time.time()
history_fe = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    verbose=1,
)
fe_time = time.time() - t0
print(f"Feature extraction time: {fe_time:.2f} sec")

## Section 4: Fine-Tuning

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable = sum(int(layer.trainable) for layer in base_model.layers)
print(f"Trainable base layers: {trainable} / {len(base_model.layers)}")

model.compile(
    optimizer=Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

print("Starting fine-tuning (last 30 layers unfrozen)...")
t0 = time.time()
history_ft = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    verbose=1,
)
ft_time = time.time() - t0
print(f"Fine-tuning time: {ft_time:.2f} sec")

## Section 5: Benchmark Both Approaches

In [ ]:
import csv

import matplotlib.pyplot as plt

results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

fe_acc = max(history_fe.history["val_accuracy"])
ft_acc = max(history_ft.history["val_accuracy"])

print("=== Benchmark ===")
print(f"Feature extraction  val_acc={fe_acc:.4f}  time={fe_time:.2f}s")
print(f"Fine-tuning         val_acc={ft_acc:.4f}  time={ft_time:.2f}s")

csv_path = results_dir / "benchmark_comparison.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Approach", "Val Accuracy", "Training Time (sec)"])
    writer.writerow(["Feature Extraction", f"{fe_acc:.4f}", f"{fe_time:.2f}"])
    writer.writerow(["Fine-Tuning", f"{ft_acc:.4f}", f"{ft_time:.2f}"])
print("Saved:", csv_path)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    ["Feature Extraction", "Fine-Tuning"],
    [fe_acc, ft_acc],
    color=["#d95f02", "#1b9e77"],
)
ax.set_ylim(0, 1)
ax.set_ylabel("Validation Accuracy")
ax.set_title("Accuracy Comparison")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
acc_plot = results_dir / "accuracy_comparison.png"
fig.savefig(acc_plot, dpi=120, bbox_inches="tight")
plt.close(fig)
print("Saved:", acc_plot)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    ["Feature Extraction", "Fine-Tuning"],
    [fe_time, ft_time],
    color=["#7570b3", "#1b9e77"],
)
ax.set_ylabel("Training Time (sec)")
ax.set_title("Time Comparison")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
time_plot = results_dir / "time_comparison.png"
fig.savefig(time_plot, dpi=120, bbox_inches="tight")
plt.close(fig)
print("Saved:", time_plot)

## Section 6: Evaluate on Test Set

In [ ]:
test_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_generator = test_datagen.flow_from_directory(
    project_root / "dataset" / "test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"Fine-tuned test accuracy: {test_acc:.4f}")
print(f"Fine-tuned test loss: {test_loss:.4f}")

## Section 7: Save Best Model

In [ ]:
models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True)
model_path = models_dir / "resnet50_finetuned.keras"
model.save(model_path)

print(f"Saved model to {model_path} (Keras .keras format)")
print("Transfer Learning Pipeline Completed")
print("Feature Extraction Benchmarked")
print("Fine-Tuning Benchmarked")
print("Best Model Saved")

## Section 8: Results Summary

Fill this table from `results/benchmark_comparison.csv` after a full run:

| Approach | Val Accuracy | Training Time (sec) |
| --- | --- | --- |
| Feature Extraction | *(from run)* | *(from run)* |
| Fine-Tuning | *(from run)* | *(from run)* |

### Completion Checklist

- [x] Transfer Learning Pipeline Completed
- [x] Feature Extraction Benchmarked
- [x] Fine-Tuning Benchmarked
- [x] Best Model Saved

## Section 9: Package Results for Download (Colab-specific)

In [ ]:
import zipfile

zip_path = project_root / "outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in (project_root / "models", project_root / "results"):
        if not folder.exists():
            continue
        for file_path in folder.rglob("*"):
            if file_path.is_file() and file_path.name != ".gitkeep":
                zf.write(file_path, file_path.relative_to(project_root))

print("Created:", zip_path.resolve())

if IN_COLAB:
    from google.colab import files

    files.download(str(zip_path))
else:
    print("Not in Colab — download/copy the zip from the path above.")